## Model#3 - Gradient Boosting

In [1]:
import sys

sys.path.append("../src")

In [2]:
from preprocessing import (
    load_and_clean_data,
    prepare_features,
    create_preprocessor
)

from evaluate import (
    evaluate_model,
    print_detailed_report
)

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier

import pandas as pd
import joblib


In [4]:
df = load_and_clean_data(
    "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
X, y = prepare_features(df)

print("X:", X.shape)
print("y:", y.shape)

X: (7032, 19)
y: (7032,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (5625, 19)
Testing: (1407, 19)


In [7]:
preprocessor = create_preprocessor(
    X_train
)

In [8]:
print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='str')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'Paper

In [9]:
gradient_boosting_model = GradientBoostingClassifier(
    random_state=42
)

In [10]:
gradient_boosting_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            gradient_boosting_model
        )
    ]
)

In [11]:
gradient_boosting_pipeline.fit(
    X_train,
    y_train
)

print("Gradient Boosting training completed!")

Gradient Boosting training completed!


In [12]:
gb_pred = gradient_boosting_pipeline.predict(
    X_test
)

print(gb_pred[:20])

[0 1 0 0 0 0 0 0 1 0 1 0 1 0 0 1 1 1 0 0]


In [13]:
gb_results = evaluate_model(
    "Gradient Boosting",
    y_test,
    gb_pred
)

gb_results

{'Model': 'Gradient Boosting',
 'Accuracy': 0.7967306325515281,
 'Precision': 0.6428571428571429,
 'Recall': 0.5294117647058824,
 'F1': 0.5806451612903226}

In [14]:
print_detailed_report(
    y_test,
    gb_pred
)

              precision    recall  f1-score   support

    No Churn       0.84      0.89      0.87      1033
       Churn       0.64      0.53      0.58       374

    accuracy                           0.80      1407
   macro avg       0.74      0.71      0.72      1407
weighted avg       0.79      0.80      0.79      1407

Confusion Matrix:
[[923 110]
 [176 198]]


In [15]:
joblib.dump(
    gradient_boosting_pipeline,
    "../models/gradient_boosting.joblib"
)

print("Gradient Boosting model saved!")

Gradient Boosting model saved!


In [16]:
results_df = pd.read_csv(
    "../reports/model_results.csv"
)
results_df

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.796309,0.634675,0.548128,0.588235
1,Random Forest,0.787491,0.630662,0.483957,0.547655


In [17]:
# Load existing model results
import pandas as pd

results_file = "../reports/model_results.csv"

results_df = pd.read_csv(results_file)

In [18]:
# Add new Gradient Boosting result
gb_df = pd.DataFrame([gb_results])

results_df = pd.concat(
    [results_df, gb_df],
    ignore_index=True
)

In [19]:
import pandas as pd
import os

results_file = "../reports/model_results.csv"

gb_df = pd.DataFrame([gb_results])

if os.path.exists(results_file):
    results_df = pd.read_csv(results_file)

    # Remove old Gradient Boosting row if it already exists
    results_df = results_df[
        results_df["Model"] != "Gradient Boosting"
    ]

    # Append latest Gradient Boosting results
    results_df = pd.concat(
        [results_df, gb_df],
        ignore_index=True
    )
else:
    results_df = gb_df

results_df.to_csv(
    results_file,
    index=False
)

print("Saved to:", results_file)
print(results_df)

Saved to: ../reports/model_results.csv
                 Model  Accuracy  Precision    Recall        F1
0  Logistic Regression  0.796309   0.634675  0.548128  0.588235
1        Random Forest  0.787491   0.630662  0.483957  0.547655
2    Gradient Boosting  0.796731   0.642857  0.529412  0.580645


In [20]:
check = pd.read_csv("../reports/model_results.csv")
check

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.796309,0.634675,0.548128,0.588235
1,Random Forest,0.787491,0.630662,0.483957,0.547655
2,Gradient Boosting,0.796731,0.642857,0.529412,0.580645


In [21]:
import os
print(os.getcwd())
print(os.path.abspath("../reports/model_results.csv"))

c:\Users\HP\ml-project\notebooks
c:\Users\HP\ml-project\reports\model_results.csv


In [22]:
print(gb_results)
print(os.path.abspath("../reports/model_results.csv"))

{'Model': 'Gradient Boosting', 'Accuracy': 0.7967306325515281, 'Precision': 0.6428571428571429, 'Recall': 0.5294117647058824, 'F1': 0.5806451612903226}
c:\Users\HP\ml-project\reports\model_results.csv
